# Imports/Settings

In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
# 1. Стандартная библиотека
import sys
import warnings
from pathlib import Path
import joblib

# 2. Сторонние библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from hydra import initialize, compose

# 3. Локальные модули
sys.path.append(str(Path.cwd().parent))
from src.core.data import UniversalDataLoader
from src.core.models import get_model

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=UserWarning, module='shap')
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
with initialize(version_base=None, config_path="../configs"):
    cfg = compose(config_name="config")

print(f"Проект: {cfg.project_name} | Режим: Error Analysis")

# --- НАСТРОЙКИ ВИЗУАЛИЗАЦИИ ---
try:
    p_cfg = cfg.logging.plots
    plt.style.use(p_cfg.style)
    plt.rcParams.update({
        'figure.figsize': p_cfg.fig_size,
        'figure.dpi': p_cfg.dpi,
        'font.size': p_cfg.font_size,
        'axes.grid': p_cfg.grid,
        'axes.spines.top': p_cfg.spines_top,
        'axes.spines.right': p_cfg.spines_right
    })
    PLOT_ALPHA = p_cfg.alpha
except AttributeError:
    PLOT_ALPHA = 0.5
    print("Используются дефолтные стили Matplotlib.")

Проект: outsource_project_name | Режим: Error Analysis


# Data Loading

In [ ]:
loader = UniversalDataLoader(cfg)
df = loader.load_data()
_, val_df, _ = loader.get_splits(df)

target = cfg.data.tabular.target_col
X_val, y_val = val_df.drop(columns=[target]), val_df[target]

# Пути к сохраненным объектам
models_dir = Path(cfg.paths.models_dir)
version = cfg.model.version
model_name = cfg.model.name

print("Загрузка препроцессора и пайплайна модели...")
prep = joblib.load(models_dir / f"preprocessing_v{version}.pkl")

model_wrapper = get_model(cfg)
model_wrapper.load(str(models_dir / f"{model_name}_v{version}.cbm"))

# Трансформируем признаки (SHAP требует матрицу, которую видела сама модель на входе в fit)
X_val_clean = prep.transform(X_val)

# Превращаем в DataFrame с правильными именами колонок после препроцессинга
if hasattr(prep, 'get_feature_names_out'):
    feature_names = prep.get_feature_names_out()
    X_val_clean_df = pd.DataFrame(X_val_clean, columns=feature_names)
else:
    X_val_clean_df = pd.DataFrame(X_val_clean, columns=X_val.columns)

print(f"Данные готовы к расчету SHAP-значений. Формат: {X_val_clean_df.shape}")

# SHAP Explainer init

In [ ]:
print("Инициализация TreeExplainer...")

# Достаем чистую нативную модель из нашей ООП-обертки
native_model = model_wrapper.model

explainer = shap.TreeExplainer(native_model)

print("Расчет SHAP-значений для валидационной выборки...")
# Чтобы не ждать слишком долго на гигантских датасетах,
# можно взять подмножество, например: X_val_clean_df.head(1000)
shap_values = explainer(X_val_clean_df)

print("Расчет успешно завершен!")

# Global interpretation

In [ ]:
plt.figure()

# Beeswarm plot показывает распределение влияния каждого признака
# Каждая точка — это один юзер/сессия.
# Направление вправо — увеличивает предсказание, влево — уменьшает.
shap.plots.beeswarm(shap_values, max_display=15, show=False)

plt.title("Глобальное влияние признаков на предсказание модели (Top 15)", fontsize=14, pad=20)
plt.tight_layout()

# Сохраняем график в папку отчетов, путь к которой берем строго из твоего paths.yaml
reports_dir = Path(cfg.paths.reports_dir)
reports_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(reports_dir / "shap_global_summary.png", bbox_inches='tight')

plt.show()

# Local Interpretation

In [ ]:
# Выбираем индекс конкретного наблюдения для анализа (например, нулевой юзер в выборке)
sample_index = 0

plt.figure()

# Waterfall plot наглядно показывает стартовую базовую вероятность (E[f(X)])
# и то, как каждый фактор шаг за шагом прибавил или отнял проценты до финального ответа.
shap.plots.waterfall(shap_values[sample_index], max_display=10, show=False)

plt.title(f"Обоснование предсказания для объекта с индексом {sample_index}", fontsize=14, pad=20)
plt.tight_layout()

# Сохраняем локальный отчет
plt.savefig(reports_dir / f"shap_local_user_{sample_index}.png", bbox_inches='tight')
plt.show()

print(f"Фактическое значение таргета для этого объекта: {y_val.iloc[sample_index]}")